# Train

For every config in `CONFIGS`, in order: train on the train split (validating on val after every epoch), evaluate the best checkpoint on the test split, and archive this executed notebook to `runs/<exp_id>/notebook.html`. Checkpoints go to `runs/<exp_id>/checkpoints/` on Drive every epoch. **After a disconnect or a GPU-quota stop, re-run every cell: a finished run trains nothing again, an interrupted one resumes from its last finished epoch, and evaluation resumes per patient.** Needs a GPU runtime (Runtime → Change runtime type → GPU). The first run of a session spends about half an hour caching the patients to local disk.

`04_evaluate.ipynb` re-scores runs and baselines on their own.

In [ ]:
# 1 — bootstrap: fresh code from GitHub (re-run after every push)
!git clone -q https://github.com/sezaiemrekonuk/ct-guided-pet-interp.git /content/petinterp 2>/dev/null || git -C /content/petinterp pull -q
%pip install -q SimpleITK monai pytorch-msssim
import sys; sys.path.insert(0, '/content/petinterp/src')
%cd /content/petinterp

In [ ]:
# 2 — runs, in order (the k=2 seed replicates still to finish)
import petinterp.runtime as rt
CONFIGS = ['configs/p4_unet_noct_k2_s2.yaml', 'configs/p4_unet_k2_s3.yaml', 'configs/p4_unet_noct_k2_s3.yaml']

In [ ]:
# 3 — train, evaluate, archive each run (resumes wherever it stopped)
from petinterp.train import train
from petinterp.evaluation.run import evaluate
for c in CONFIGS:
    run = rt.start(c)
    train(run)
    evaluate(run)
    rt.archive_notebook(run.dir)